# Fine-Tuning di Llama 3.1 8B con LoRA per l'Information Extraction
Questo notebook documenta il processo di **Supervised Fine-Tuning (SFT)** applicato a un LLM (Llama 3.1 8B) per un task di **Information Extraction** nel dominio culinario. 
L'obiettivo è duplice (sotto-task simultanei):
1. **Named Entity Recognition (NER)**: Identificazione delle entità (es. ingredienti, piatti, tecniche).
2. **Relation Extraction (RE)**: Identificazione dei legami semantici per popolare un Knowledge Graph sotto forma di triple `Soggetto | Relazione | Oggetto`.

Si adotta la tecnica **PEFT (Parameter-Efficient Fine-Tuning)** tramite **LoRA (Low-Rank Adaptation)** per aggiornare solo una frazione minima di parametri, ottimizzando le risorse computazionali.

## 1. Setup dell'Ambiente e Installazione delle Dipendenze
In questa sezione installiamo il framework **Unsloth**, ottimizzato per accelerare il fine-tuning di LLM su GPU singola

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl datasets accelerate bitsandbytes

## 2. Configurazione, Caricamento Dati e Inizializzazione del Modello
In questa fase configuriamo gli iperparametri specidici, carichiamo il dataset (da noi creato) in formato JSONL.
Il modello base `Llama-3.1-8b` viene caricato in modalità **4-bit quantization (NF4)** per ridurre drasticamente l'occupazione di VRAM. Successivamente, iniettiamo il Chat Template ufficiale per strutturare correttamente i token speciali di sistema e conversazione.

In [ ]:
import os
import torch
import re
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig


N_EPOCHS = 3  
LR = 2e-4
MAX_LEN = 2048

MODEL_CHECKPOINT = "unsloth/meta-llama-3.1-8b-bnb-4bit"
DATASET_FILE = "/kaggle/input/datasets/vincenzorubulotta/dataset-triple-culinarie/dataset_triple_culinarie.jsonl"


dataset = load_dataset("json", data_files=DATASET_FILE)["train"]
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split["train"]
val_dataset = dataset_split["test"]


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_CHECKPOINT,
    max_seq_length = MAX_LEN,
    dtype = None, # Autodetect (BF16/FP16)
    load_in_4bit = True,
)

# iniettare il Chat Template ufficiale di Llama 3 nel Tokenizer
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% set loop_messages = messages %}"
        "{% for message in loop_messages %}"
        "{{ '<|start_header_id|>' + message['role'] + '<|end_header_id|>\\n\\n' + message['content'] | trim + '<|eot_id|>' }}"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "{{ '<|start_header_id|>assistant<|end_header_id|>\\n\\n' }}"
        "{% endif %}"
    )


model_lora = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0, 
    bias = "none",
    random_state = 3407,
)


def applica_prompt_template(esempio):
    messaggi = [
        {"role": "system", "content": esempio['instruction']},
        {"role": "user", "content": esempio['input']},
        {"role": "assistant", "content": esempio['output']}
    ]
    esempio["text"] = tokenizer.apply_chat_template(messaggi, tokenize=False, add_generation_prompt=False)
    return esempio

train_dataset = train_dataset.map(applica_prompt_template)
val_dataset = val_dataset.map(applica_prompt_template)

#configurazione trainer
sft_config = SFTConfig(
    output_dir="./culinary_lora_output",
    learning_rate=LR,
    per_device_train_batch_size=2, 
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    num_train_epochs=N_EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=5,
    bf16=False, 
    fp16=True,
    max_grad_norm=0.3,
    weight_decay=0.01, 
    optim="paged_adamw_8bit",
    save_total_limit=1,
    report_to="none",
    dataset_text_field="text"
)

#inizializzazione trainer
trainer = SFTTrainer(
    model=model_lora,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer, 
    args=sft_config
)

print("🏋️‍♂️ Avvio dell'addestramento...")
trainer.train()

## 2.1 Analisi delle Metriche di Addestramento 
Di seguito vengono riportati i risultati dell'andamento della Loss registrati durante le 3 epoche di addestramento. 

L'analisi della curva di **Training Loss** e della **Validation Loss** permette di verificare la convergenza del modello e l'assenza di fenomeni di overfitting (visto il tracking regolare ad ogni epoca).
<img src="training_loss.png" alt="GRAFICO" width="700">


## 3. Inferenza e Post-Processing dell'Output
Una volta completato il fine-tuning, impostiamo il modello in modalità `for_inference` (disattivando i meccanismi di dropout e abilitando il caching dei k-v tensori). 

L'output testuale viene poi pulito tramite espressioni regolari per isolare ed estrarre esclusivamente le triple sintatticamente valide.

In [ ]:
FastLanguageModel.for_inference(model_lora)

#Usiamo come ricetta un Post da noi generato con il sistema agentico
nuova_ricetta = """La cucina siciliana è famosa per le sue ricette deliziose e autentiche, e uno dei piatti più rappresentativi di questa tradizione è senza dubbio la "Norma". Questo piatto, che prende il nome dall'opera lirica di Vincenzo Bellini, è un vero e proprio tributo alla cucina siciliana e ai suoi ingredienti tipici. In questo articolo, vi guiderò nella preparazione dei Tortiglioni alla Norma, un piatto che combina la pasta con le melanzane, il pomodoro e la ricotta salata.

Per iniziare, è importante avere a disposizione gli ingredienti necessari. Ecco la lista degli ingredienti obbligatori per la preparazione dei Tortiglioni alla Norma:

* Melanzane (1 grande) (Circa 0.65 € (per 0.5 Kg))
* Pomodori pelati (400 g) (Circa 1.08 € (per 0.5 Kg))
* Basilico (q.b.) (Circa 0.90 € (per 0.05 Kg))
* Olio extravergine d'oliva (q.b.) (Circa 0.57 € (per 0.05 Kg))
* Aglio (2 spicchi) (Circa 0.20 € (per 0.05 Kg))
* Sale (q.b.) (Circa 0.004 € (per 0.005 Kg))
* Pepe nero (q.b.) (Circa 0.175 € (per 0.005 Kg))
* Ricotta salata (200 g) (Circa 0.43 € (per 0.05 Kg))
* Tortiglioni (400 g) (Circa 0.19 € (per 0.05 Kg))
* Parmigiano Reggiano DOP (100 g) (Circa 1.10 € (per 0.05 Kg))

La preparazione dei Tortiglioni alla Norma inizia con la pulizia e il taglio delle melanzane. È importante tagliare le melanzane a fette sottili e lasciarle riposare per circa 30 minuti in modo che perdano il loro eccesso di acqua [Fonte: https://cookingitalians.com/authentic-pasta-alla-norma-a-taste-of-sicily/]. Successivamente, le melanzane vanno fritte in abbondante olio caldo fino a quando non sono dorate [Fonte: Knowledge Graph].

Mentre le melanzane sono in cottura, possiamo preparare il sugo. In un grande tegame, soffriggiamo l'aglio e l'olio, poi aggiungiamo la passata di pomodoro e cuociamo per circa 20 minuti [Fonte: https://cookingitalians.com/authentic-pasta-alla-norma-a-taste-of-sicily/]. Infine, aggiungiamo il basilico fresco e il sale [Fonte: Knowledge Graph].

La pasta, ovviamente, è un elemento fondamentale di questo piatto. I Tortiglioni vanno cotti al dente in abbondante acqua salata, poi scolati e uniti al sugo [Fonte: Knowledge Graph]. A questo punto, aggiungiamo le melanzane fritte e la ricotta salata grattugiata sopra [Fonte: https://cookingitalians.com/authentic-pasta-alla-norma-a-taste-of-sicily/].

È importante non cuocere troppo la pasta e non utilizzare mozzarella o altri formaggi filanti, in quanto ciò altererebbe il sapore tradizionale della ricetta [Fonte: https://cookingitalians.com/authentic-pasta-alla-norma-a-taste-of-sicily/]. La Norma è un piatto che richiede semplicità e autenticità, quindi è fondamentale rispettare le tradizioni culinarie siciliane.

In conclusione, i Tortiglioni alla Norma sono un piatto delizioso e autentico che rappresenta al meglio la cucina siciliana. Con gli ingredienti giusti e la preparazione appropriata, potrete gustare un piatto che è stato apprezzato per generazioni.

💡 I Consigli dello Chef
Se desiderate una variante più sostenibile e naturale, potete sostituire gli ingredienti con le loro alternative BIO. Ad esempio, potete utilizzare melanzane BIO, pomodori BIO, basilico BIO, olio extravergine di oliva BIO, aglio BIO, sale fino BIO, pepe nero BIO, ricotta BIO, pasta di semola BIO e parmigiano reggiano BIO. Inoltre, se siete intolleranti al lattosio, potete sostituire la ricotta salata con una variante senza lattosio.

🍷 L'Abbinamento Perfetto
Chianti Classico, un vino rosso toscano con note di frutta rossa e spezie, che si abbina bene con i sapori decisi della ricetta."""


messaggi_test = [
    {
        "role": "system", 
        "content": "Analizza il testo della ricetta ed estrai le relazioni nel formato rigido: Soggetto | RELAZIONE | Oggetto. Usa solo i termini approvati: USA_INGREDIENTE, USA_TECNICA, TIPO_DI_PIATTO."
    },
    {
        "role": "user", 
        "content": nuova_ricetta
    }
]


prompt_test = tokenizer.apply_chat_template(messaggi_test, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([prompt_test], return_tensors="pt").to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>"),
    tokenizer.convert_tokens_to_ids("<|end_of_text|>") 
]

terminatori_validi = [t for t in terminators if t is not None]


outputs = model_lora.generate(
    **inputs, 
    max_new_tokens = 512, 
    use_cache = True,
    eos_token_id = terminatori_validi, 
    pad_token_id = tokenizer.pad_token_id,
    repetition_penalty = 1.2, 
    do_sample = False 
)


# PULIAMO L'OUTPUT 
risposta_pulita = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

righe_valide = []
for riga in risposta_pulita.split('\n'):

    if riga.count('|') == 2:
        
      
        parti = riga.split('|')
        soggetto = parti[0].strip()
        relazione = parti[1].strip()
        oggetto = parti[2].strip()
        oggetto = re.sub(r'[^\x00-\x7F\xC0-\xFF]', '', oggetto)
        oggetto = re.sub(r'(?i)\bassistant\b', '', oggetto)
        oggetto = re.sub(r'\b\w*([a-zA-Z])\1{2,}\w*\b', '', oggetto)
        oggetto = oggetto.strip()
 
        if oggetto:
            tripla_pulita = f"{soggetto} | {relazione} | {oggetto}"
            righe_valide.append(tripla_pulita)
    else:
        break 

print("\n--- TRIPLE ESTRATTE E PULITE ---")
print('\n'.join(righe_valide))

## 4. Serializzazione e Download dei Pesi del Modello
In quest'ultima sezione salviamo localmente i pesi appresi dall'adattatore LoRA e i file di configurazione del tokenizer. Infine, comprimiamo la cartella in un archivio `.zip` e generiamo un link per il download diretto del modello pronto per il deployment.

In [ ]:
import os
import shutil
from IPython.display import FileLink

model_lora.save_pretrained("./mio_modello_lora_final")
tokenizer.save_pretrained("./mio_modello_lora_final")

shutil.make_archive("il_mio_modello_lora", 'zip', "./mio_modello_lora_final")

print("\n✅ Link pronto! Clicca qui sotto per scaricare il file zip:")
display(FileLink("il_mio_modello_lora.zip"))